# Value-Based Reinforcement Learning

## Q-Learning

Instead of asking "what probability should I assign to each action?", we ask a different question:

- How valuable is taking action $a$ in state $s$?

That is: $Q(s, a)$.

### Example

Suppose a robot is at an intersection.

| Action | Q-value |
|---|---:|
| left | 3.2 |
| right | 7.8 |
| forward | 5.1 |

The agent simply chooses:

$$
\arg\max_a Q(s, a) \to \text{right}
$$

## 1. Where does Q come from?

We do not know the correct Q-values initially. We start with zeros:

```text
Q-Table

        LEFT    RIGHT
S0      0       0
S1      0       0
...
```

Then we interact with the environment and update.

### Example trajectory

Suppose:

```text
S0 --RIGHT--> reward: +1 --> S1
```

We want $Q(S_0, \text{RIGHT})$ to reflect:

$$
\text{immediate reward} + \text{future value}
$$

The Q-learning update is:

$$
Q(s,a) \gets Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]
$$

The key idea: compare the current estimate against the reward plus the best estimated future, then move the current estimate toward that target.

## 2. This is another TD method

Look at the target:

$$
r + \gamma \max_{a'} Q(s', a')
$$

This is the same bootstrapping trick as TD learning:

- Use the estimated future instead of waiting for the whole episode.
- But now we estimate action values rather than state values.

## 3. Why max?

Suppose after reaching $S_1$:

| Action | Q-value |
|---|---:|
| LEFT | 4 |
| RIGHT | 8 |
| FORWARD | 3 |

The best future value is:

$$
\max_a Q(S_1, a) = 8
$$

We are asking: "If I reach this next state, what is the best future I can get?"

That is why Q-learning is naturally value-based.

## 4. Exploration

There is an immediate problem.

If we always do:

$$
\text{action} = \arg\max Q[\text{state}]
$$

then initially all values are identical, and the agent might pick one action forever and never discover anything better.

### ε-greedy strategy

With probability $\epsilon$, take a random action. Otherwise, take the greedy action:

$$
\text{action} = \begin{cases}
\text{random} & \text{with probability } \epsilon \\
\arg\max Q(s,a) & \text{otherwise}
\end{cases}
$$

Example with $\epsilon = 0.2$:

- 20% of the time: explore randomly
- 80% of the time: exploit the best known action

### Epsilon decay

Usually we decay $\epsilon$ over time:

- Early training: $\epsilon = 1.0$ (explore heavily)
- Later: $\epsilon = 0.05$ (mostly exploit)

## Grid World Environment

### Layout

```text
S . . .
. . X .
. . . .
. . . G
```

Where:

- **S** = start position
- **G** = goal position
- **X** = obstacle

### Actions

- 0 = up
- 1 = down
- 2 = left
- 3 = right

### Reward Structure

| Outcome | Reward |
|---|---:|
| Reach goal | +10 |
| Normal step | -0.1 |
| Hit obstacle | -1 |
| Hit boundary | -1 |

The agent must discover the optimal path to the goal while minimizing step penalties.

In [7]:
import random
import numpy as np


GRID_SIZE = 4
ACTIONS = 4

ALPHA = 0.1
GAMMA = 0.99

EPISODES = 5000

EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995


q_table = np.zeros(
    (GRID_SIZE, GRID_SIZE, ACTIONS),
    dtype=np.float32,
)


START = (0, 0)
GOAL = (3, 3)
OBSTACLE = (1, 2)


def reset():
    return START


def step(state, action):

    row, col = state

    next_row = row
    next_col = col

    if action == 0:
        next_row -= 1
    elif action == 1:
        next_row += 1
    elif action == 2:
        next_col -= 1
    elif action == 3:
        next_col += 1

    # Boundary
    if not (
        0 <= next_row < GRID_SIZE
        and 0 <= next_col < GRID_SIZE
    ):
        return state, -1, False

    next_state = (next_row, next_col)

    # Obstacle
    if next_state == OBSTACLE:
        return state, -1, False

    # Goal
    if next_state == GOAL:
        return next_state, 10, True

    return next_state, -0.1, False


def choose_action(state, epsilon):

    if random.random() < epsilon:
        return random.randrange(ACTIONS)

    row, col = state

    return int(
        np.argmax(q_table[row, col])
    )


epsilon = EPSILON


for episode in range(EPISODES):

    state = reset()

    for step_count in range(100):

        action = choose_action(
            state,
            epsilon,
        )

        next_state, reward, done = step(
            state,
            action,
        )

        row, col = state

        next_row, next_col = next_state

        current_q = q_table[
            row,
            col,
            action,
        ]

        if done:

            target = reward

        else:

            target = (
                reward
                + GAMMA
                * np.max(
                    q_table[
                        next_row,
                        next_col,
                    ]
                )
            )

        q_table[
            row,
            col,
            action,
        ] = current_q + ALPHA * (
            target - current_q
        )

        state = next_state

        if done:
            break

    epsilon = max(
        EPSILON_MIN,
        epsilon * EPSILON_DECAY,
    )


print("\nLearned policy:\n")

symbols = {
    0: "↑",
    1: "↓",
    2: "←",
    3: "→",
}

for row in range(GRID_SIZE):

    line = ""

    for col in range(GRID_SIZE):

        if (row, col) == START:
            line += " S "

        elif (row, col) == GOAL:
            line += " G "

        elif (row, col) == OBSTACLE:
            line += " X "

        else:

            action = np.argmax(
                q_table[row, col]
            )

            line += f" {symbols[action]} "

    print(line)


Learned policy:

 S  →  →  ↓ 
 ↑  ↓  X  ↓ 
 →  →  →  ↓ 
 →  ↑  →  G 


## Understanding Gamma (Discount Factor)

The parameter $\gamma$ controls how much the agent cares about future rewards.

In Q-learning:

$$
Q \gets r + \gamma \max Q(s', a')
$$

We used $\gamma = 0.99$.

### Effect of $\gamma$

| Value | Interpretation |
|---|---|
| $\gamma \approx 0$ | Care mostly about immediate reward |
| $\gamma \approx 1$ | Care strongly about long-term reward |

### In our grid example

With $\gamma = 0.1$:

- The agent asks: "Is this next move good?"
- It focuses on immediate rewards.

With $\gamma = 0.99$:

- The agent asks: "Will this move eventually get me to the goal?"
- It can learn longer paths toward the +10 goal despite small -0.1 step penalties.

That is why a high discount factor is necessary for learning long-horizon strategies.

## DQN: Q-Table to Neural Network

We already understand $Q(s,a)$ and how Q-learning updates it. But there is a fundamental problem with Q-tables.

### The Problem with Q-Tables

The Q-table size grows exponentially with state space.

| Environment | State Space Size | Q-Table Size |
|---|---|---|
| Small grid | 16 states × 4 actions | Manageable |
| Image-based (84×84×RGB) | Millions of states | Impossible |

For any realistic environment, a table is completely useless.

### The DQN Solution

Can a neural network approximate $Q(s,a)$? **Yes.**

## 1. The DQN Architecture

Instead of a lookup table:

```text
state → Q-table → Q(s,a)
```

We use a neural network:

```text
state → Q-network → [Q(s,a₀), Q(s,a₁), Q(s,a₂), ...]
```

### Example

For our 4-action grid:

```text
state (row, col)
       ↓
  Q-network
       ↓
[1.2, 4.8, 0.3, 7.1]
       ↑
     best action is index 1
```

The decision rule remains the same:

$$
\text{action} = \arg\max Q_\theta(s)
$$

Only the representation of Q changed from a table to a neural network.

## 2. The DQN Target

The Q-learning target was:

$$
r + \gamma \max_a Q(s', a)
$$

With a neural network:

$$
\text{target} = r + \gamma \max_a Q_\theta(s', a)
$$

We train the network so that:

$$
Q_\theta(s, a) \approx \text{target}
$$

The loss is simply regression:

$$
L = (Q_\theta(s,a) - \text{target})^2
$$

## 3. Why DQN isn't just "Q-learning with PyTorch"

Two fundamental problems appear immediately.

### Problem 1: Correlated Data

The agent experiences:

$$
S_1 \to S_2 \to S_3 \to S_4 \to S_5
$$

These samples are highly correlated. Neural networks train better when samples are shuffled and independent.

### Problem 2: Moving Target

We use the same network to:

- predict $Q$
- construct the target $Q$

So the target keeps moving while we chase it, making training unstable.

### The DQN Solutions

DQN introduced two key mechanisms:

1. **Experience Replay** — break temporal correlation
2. **Target Network** — stabilize the target

## 4. Experience Replay

Instead of training immediately on each transition, we store it in a replay buffer:

```text
Replay Buffer: [(S₁,A,R,S₂), (S₂,A,R,S₃), ...]
```

Then randomly sample batches for training:

```text
Sample batch: [B, E, H, A, D]
              ↓
         Q-network
              ↓
         gradient update
```

### Benefits

- Breaks temporal correlation from the sequential trajectory
- Allows reusing old experiences multiple times
- Massive efficiency improvement

## 5. Target Network

We maintain two Q-networks:

| Network | Update Frequency |
|---|---|
| Online network ($Q_\theta$) | Every gradient step |
| Target network ($Q_{\theta^-}$) | Every N steps |

The target network stays relatively stable while the online network is trained constantly.

The target uses:

$$
y = r + \gamma \max_a Q_{\theta^-}(s', a)
$$

Then:

$$
L = (Q_\theta(s,a) - y)^2
$$

Every N steps:

$$
\theta^- \gets \theta
$$

### Why This Helps

- The target is relatively stable, so the network has a consistent learning signal.
- This stabilizes training massively compared to using the same network for both Q and target.

## 6. Our DQN Implementation

We use the same grid environment as before.

The state is simply: (row, col)

We encode it as normalized floats for the neural network.

In [ ]:
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn


GRID_SIZE = 4
ACTION_SIZE = 4

GAMMA = 0.99
LEARNING_RATE = 1e-3

BUFFER_SIZE = 10_000
BATCH_SIZE = 64

TARGET_UPDATE = 100

EPISODES = 1000

EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995


START = (0, 0)
GOAL = (3, 3)
OBSTACLE = (1, 2)


class GridWorld:

    def reset(self):
        return START

    def step(self, state, action):

        row, col = state

        next_row = row
        next_col = col

        if action == 0:
            next_row -= 1

        elif action == 1:
            next_row += 1

        elif action == 2:
            next_col -= 1

        elif action == 3:
            next_col += 1

        if not (
            0 <= next_row < GRID_SIZE
            and 0 <= next_col < GRID_SIZE
        ):
            return state, -1, False

        next_state = (next_row, next_col)

        if next_state == OBSTACLE:
            return state, -1, False

        if next_state == GOAL:
            return next_state, 10, True

        return next_state, -0.1, False


def encode_state(state):

    row, col = state

    return torch.tensor(
        [
            row / (GRID_SIZE - 1),
            col / (GRID_SIZE - 1),
        ],
        dtype=torch.float32,
    )


class QNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, ACTION_SIZE),
        )

    def forward(self, state):

        return self.network(state)


env = GridWorld()

online_net = QNetwork()
target_net = QNetwork()

target_net.load_state_dict(
    online_net.state_dict()
)

target_net.eval()


optimizer = torch.optim.Adam(
    online_net.parameters(),
    lr=LEARNING_RATE,
)


replay_buffer = deque(
    maxlen=BUFFER_SIZE
)


epsilon = EPSILON


def choose_action(state):

    if random.random() < epsilon:
        return random.randrange(ACTION_SIZE)

    with torch.no_grad():

        state_tensor = encode_state(
            state
        ).unsqueeze(0)

        q_values = online_net(
            state_tensor
        )

        return q_values.argmax(
            dim=1
        ).item()


def train_step():

    if len(replay_buffer) < BATCH_SIZE:
        return

    batch = random.sample(
        replay_buffer,
        BATCH_SIZE,
    )

    states, actions, rewards, next_states, dones = zip(
        *batch
    )

    states = torch.stack(
        [encode_state(s) for s in states]
    )

    next_states = torch.stack(
        [encode_state(s) for s in next_states]
    )

    actions = torch.tensor(
        actions,
        dtype=torch.long,
    )

    rewards = torch.tensor(
        rewards,
        dtype=torch.float32,
    )

    dones = torch.tensor(
        dones,
        dtype=torch.float32,
    )

    # Q(s, a)
    q_values = online_net(states)

    chosen_q = q_values.gather(
        1,
        actions.unsqueeze(1),
    ).squeeze(1)

    # Target
    with torch.no_grad():

        next_q_values = target_net(
            next_states
        )

        max_next_q = next_q_values.max(
            dim=1
        ).values

        targets = (
            rewards
            + GAMMA
            * max_next_q
            * (1 - dones)
        )

    loss = nn.functional.mse_loss(
        chosen_q,
        targets,
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()


step_count = 0


for episode in range(EPISODES):

    state = env.reset()

    total_reward = 0

    for _ in range(100):

        action = choose_action(state)

        next_state, reward, done = env.step(
            state,
            action,
        )

        replay_buffer.append(
            (
                state,
                action,
                reward,
                next_state,
                done,
            )
        )

        state = next_state

        total_reward += reward

        train_step()

        step_count += 1

        if step_count % TARGET_UPDATE == 0:

            target_net.load_state_dict(
                online_net.state_dict()
            )

        if done:
            break

    epsilon = max(
        EPSILON_MIN,
        epsilon * EPSILON_DECAY,
    )

    if (episode + 1) % 100 == 0:

        print(
            f"Episode {episode + 1:4d} | "
            f"Reward {total_reward:6.2f} | "
            f"Epsilon {epsilon:.3f}"
        )